## Conclusion de la Phase 2 : les liaisons durablement en difficulté

**Question** : quelles liaisons sont durablement en difficulté, comment évoluent-elles, et quelle cause porte cette évolution ?

### Méthode
- Une ligne par liaison × année, avec des **années d'octobre à septembre** pour coller aux périodes du cadrage : référence = 2023, 2024, 2025 (oct. 2022 → sept. 2025) ; suivi = 2026 (oct. 2025 → sept. 2026).
- Taux de retard = trains en retard à l'arrivée ÷ trains prévus, pondéré par le nombre de trains.
- Une classe de durée par liaison (durée médiane) : 20 liaisons de moins de 1 h 30, 58 de 1 h 30 à 3 h, 52 de plus de 3 h.
- **Seuil** = quartile supérieur du taux de retard dans chaque classe : 18,1 % (< 1h30), 16,4 % (1h30-3h), 17,7 % (> 3h).
- **Volume minimal** = 100 trains par an (75 pour 2018, incomplète), choisi pour la stabilité du taux : 9 liaisons-années écartées sur 1 116, toutes marginales.
- **Durablement en difficulté** = au-dessus du seuil au moins 2 années sur 3 de la période de référence.

### Résultats
1. **26 liaisons durablement en difficulté** sur 121 (21 %), dont 18 au-dessus du seuil les 3 années. 5 sont internationales, avec des données suspectes. On retrouve deux pôles : **Lyon Part-Dieu** (7 liaisons, surtout province-province) et l'**axe Paris – Sud-Est** (Mâcon, Chambéry, Valence, Nîmes, Annecy). Plusieurs liaisons sont sélectionnées dans les deux sens : la difficulté touche des axes.
2. **2026 est une année dégradée pour tout le réseau** : +4 points de taux de retard pour la liaison médiane, 97 % des liaisons touchées, avec des pics en janvier et de mai à août. Les liaisons sélectionnées ne se dégradent qu'un peu plus (+4,7 points). Par rapport à cette tendance, 12 se dégradent davantage (dont 9 françaises), 7 suivent le réseau, 7 résistent mieux.
3. **La dégradation de 2026 vient des Voyageurs et de l'Externe** (+2,8 et +2,2 points pour la liaison médiane), pas du Réseau (−1,1 point).
4. **Exception : l'axe Paris – Mâcon – Chambéry**. Ses trois liaisons sont les plus forts décrochages français, et leur cause propre est le Réseau, alors qu'il recule ailleurs. Italie → Paris Lyon, sur le même couloir, suit le même schéma. Origine à vérifier.

### Limites
- Les liaisons internationales concentrent les anomalies (taux à 0 % ou supérieurs à 50 %, causes manquantes).
- Une liaison sélectionnée pour ses mauvais taux tend à se rapprocher de la moyenne l'année suivante (régression vers la moyenne) : les « améliorations » relatives sont à lire avec prudence.
- Une seule cause est retenue par train : la hausse des incidents externes peut faire baisser mécaniquement la part du Réseau.
- Les évolutions médianes des familles ne s'additionnent pas exactement : ce sont des tendances.
- Le seuil est calculé sur toutes les années, y compris les années atypiques (2018, 2020-2021).

### Questions pour le groupe
1. Exclure ou seulement signaler les liaisons internationales ?
2. Calculer le taux sur les trains prévus ou sur les trains circulés (prévus − annulés) ?
3. Calculer le seuil sur toutes les années ou sur la seule période de référence ?
4. Présenter les liaisons par sens ou par axe dans l'application ?

**Étape 1 : le taux de retard par liaison, par an et par classe de durée**

In [1]:
#import
import pandas as pd

from railinsight.data.loader import REGULARITE, load
from railinsight.data.quality import normaliser_noms

In [2]:
#chargement et nettoyage des noms de gare
df = load(REGULARITE)

for col in ["gare_depart", "gare_arrivee"]:
    df[col] = normaliser_noms(df[col])

print(df.shape)

(12907, 26)


**Année d'octobre à septembre**

La période de référence (oct. 2022 → sept. 2025) et la période de suivi (oct. 2025 → sept. 2026) ne suivent pas l'année civile. On utilise donc des années d'octobre à septembre, nommées par l'année où elles se terminent (oct. 2024 → sept. 2025 = 2025).
- Référence : 2023, 2024, 2025 (3 années complètes). Suivi : 2026 (complète).
- 2018 ne compte que 9 mois (le jeu commence en janvier 2018).

In [3]:
# Année d'octobre à septembre : octobre, novembre et décembre comptent pour l'année suivante
mois = df["date"].str[5:7].astype(int)
df["annee"] = df["date"].str[:4].astype(int) + (mois >= 10).astype(int)

print(df["annee"].value_counts().sort_index())

annee
2018    1170
2019    1497
2020    1476
2021    1455
2022    1463
2023    1456
2024    1444
2025    1494
2026    1452
Name: count, dtype: int64


In [4]:
#durée par liaison
#utilisation de la médiane pour els durées mensuelle, résiste mieux aux mois atypiques. 
liaison = ["gare_depart", "gare_arrivee"]

duree_liaison = (
    df.groupby(liaison)["duree_moyenne"]
    .median()
    .rename("duree_liaison")
    .reset_index()
)

print(duree_liaison.shape)
duree_liaison.head()

(130, 3)


,gare_depart,gare_arrivee,duree_liaison
0,AIX EN PROVENCE,PARIS LYON,186.0
1,ANGERS ST LAUD,PARIS MONTPARNASSE,95.0
2,ANGOULEME,PARIS MONTPARNASSE,128.0
3,ANNECY,PARIS LYON,227.0
4,ARRAS,PARIS NORD,50.0


In [5]:
#la classe de la durée
#à cause de pd.cut, une durée d'exactement 90 minutes tombe dans <1h30
duree_liaison["classe_duree"] = pd.cut(
    duree_liaison["duree_liaison"],
    bins=[0, 90, 180, float("inf")],
    labels=["< 1h30", "1h30-3h", "> 3h"],
)

print(duree_liaison["classe_duree"].value_counts())

classe_duree
1h30-3h    58
> 3h       52
< 1h30     20
Name: count, dtype: int64


In [6]:
#taux de retard par liaison et par an
par_an = (
    df.groupby(liaison + ["annee"])
    .agg(
        nb_train_prevu=("nb_train_prevu", "sum"),
        nb_train_retard_arrivee=("nb_train_retard_arrivee", "sum"),
    )
    .reset_index()
)

par_an["taux_retard"] = par_an["nb_train_retard_arrivee"] / par_an["nb_train_prevu"]

In [7]:
#ajout de la classe de trajet à chaque ligne
par_an = par_an.merge(duree_liaison, on=liaison)

print(par_an.shape)
print(par_an.groupby(liaison)["classe_duree"].nunique().max())

(1116, 8)
1


### Bilan de l'étape 1 : taux de retard par liaison, par an et par classe

- **Données** : 12 907 observations (liaison × mois), de janvier 2018 à septembre 2026.
- **Années d'octobre à septembre** : référence = 2023, 2024, 2025 ; suivi = 2026. 2018 ne compte que 9 mois (1 170 observations, contre 1 444 à 1 497 pour les autres années).
- **Toutes les observations sont gardées** : les 83 écartées en Phase 1 ne l'étaient que pour l'analyse des causes.
- **Classe de durée** : une seule par liaison, à partir de la durée médiane. 20 liaisons de moins de 1 h 30, 58 de 1 h 30 à 3 h, 52 de plus de 3 h.
- **Tableau obtenu** : 1 116 lignes liaison × année, avec trains prévus, trains en retard et taux de retard (pondéré par le nombre de trains).
- **Contrôle** : chaque liaison a une seule classe. Le notebook d'exploration, qui classait mois par mois, coupait certaines liaisons en deux.

**Étape 2 : la distribution par classe et le seuil**

In [8]:
#vérification des cas limites
#ligne 1 : liaisons-années sans aucun train prévu, ligne 2 compte les NaN de la division par 0. 

print((par_an["nb_train_prevu"] == 0).sum())
print(par_an["taux_retard"].isna().sum())

2
2


In [9]:
#distribution du taux de retard par classe
print(par_an.groupby("classe_duree", observed=True)["taux_retard"].describe().round(3))

              count   mean    std    min    25%    50%    75%    max
classe_duree                                                        
< 1h30        180.0  0.152  0.046  0.058  0.114  0.147  0.181  0.288
1h30-3h       480.0  0.138  0.051  0.045  0.102  0.126  0.164  0.346
> 3h          454.0  0.148  0.062  0.000  0.103  0.142  0.177  0.565


In [10]:
#le seuil par classe
seuils = par_an.groupby("classe_duree", observed=True)["taux_retard"].quantile(0.75)
print(seuils.round(3))

classe_duree
< 1h30     0.181
1h30-3h    0.164
> 3h       0.177
Name: taux_retard, dtype: float64


### Bilan de l'étape 2 : distribution et premiers seuils

**Cas limites** : 2 liaisons-années sans train (Madrid ↔ Marseille 2021, Covid), donc 2 taux manquants. Elles sont écartées par le volume minimal (étape 3).

| Classe | Liaisons-années | Médiane | Seuil (75 %) |
|---|---|---|---|
| < 1h30 | 180 | 14,7 % | 18,1 % |
| 1h30-3h | 480 | 12,6 % | 16,4 % |
| > 3h | 454 | 14,2 % | 17,7 % |

**Points d'attention**
- Les trajets courts ont la médiane la plus élevée, cohérent avec un seuil de ponctualité plus strict (5 min sous 1 h 30). Hypothèse à confirmer.
- La classe « < 1h30 » ne repose que sur 20 liaisons : son seuil est plus fragile.
- Les seuils mélangent toutes les années, y compris les années atypiques (2018, 2020-2021).

**Question pour le groupe** : diviser par les trains prévus ou par les trains réellement circulés (prévus − annulés) ?

**Etape 3 : le volume minimal**

In [11]:
#cas extrêmes de la classe ">3h"
colonnes = liaison + ["annee", "nb_train_prevu", "nb_train_retard_arrivee", "taux_retard"]
grands_trajets = par_an[par_an["classe_duree"] == "> 3h"]

print(grands_trajets.sort_values("taux_retard")[colonnes].head(3))
print(grands_trajets.sort_values("taux_retard")[colonnes].tail(3))

               gare_depart          gare_arrivee  annee  nb_train_prevu  \
625             PARIS LYON                ITALIE   2024             212   
364   MARSEILLE ST CHARLES                MADRID   2023              67   
1079             TOURCOING  MARSEILLE ST CHARLES   2022               2   

      nb_train_retard_arrivee  taux_retard  
625                         0          0.0  
364                         0          0.0  
1079                        0          0.0  
              gare_depart          gare_arrivee  annee  nb_train_prevu  \
528             PARIS EST             STUTTGART   2026            1570   
320                MADRID  MARSEILLE ST CHARLES   2021               0   
362  MARSEILLE ST CHARLES                MADRID   2021               0   

     nb_train_retard_arrivee  taux_retard  
528                      887     0.564968  
320                        0          NaN  
362                        0          NaN  


In [12]:
gares_etrangeres = ["BARCELONA", "MADRID", "FRANCFORT", "GENEVE", "LAUSANNE", "STUTTGART", "ZURICH", "ITALIE"]
par_an["internationale"] = par_an["gare_depart"].isin(gares_etrangeres) | par_an["gare_arrivee"].isin(gares_etrangeres)

zero_suspect = (par_an["nb_train_retard_arrivee"] == 0) & (par_an["nb_train_prevu"] >= 100)
print(zero_suspect.sum(), "liaisons-années à 0 retard avec au moins 100 trains")
print(par_an.loc[zero_suspect, "internationale"].value_counts())

2 liaisons-années à 0 retard avec au moins 100 trains
internationale
True    2
Name: count, dtype: int64


In [13]:
#On retire 2018 qui n'est pas une année pleine
annees_pleines = par_an[par_an["annee"] > 2018]

print(annees_pleines["nb_train_prevu"].describe().round(0))
print(annees_pleines["nb_train_prevu"].quantile([0.05, 0.10, 0.20]))

# on cherche un seuil qui écarte les vrais petits volumes (les 2 trains Tourcoing-Marseille) et non les liaisons à 1 train par jour 

for seuil in [50, 100, 200]:
    print(seuil, "trains :", (annees_pleines["nb_train_prevu"] < seuil).sum(), "liaisons-années")

count      986.0
mean      3261.0
std       2178.0
min          0.0
25%       1842.0
50%       2710.0
75%       4280.0
max      12857.0
Name: nb_train_prevu, dtype: float64
0.05     464.75
0.10    1001.00
0.20    1604.00
Name: nb_train_prevu, dtype: float64
50 trains : 3 liaisons-années
100 trains : 9 liaisons-années
200 trains : 16 liaisons-années


Le problème est qu'on veut repérer les liaisons qui ont trop de retard. Mais pour que ce chiffre soit valable, il faut qu'il y ait assez de train. il faut donc écarter les liaisons-années qui ont trop peu de trains. 

Deux façons de fixer la barre :

Par rang : exclure les 10 % les plus petites. Ça donne 1 046 trains. Problème : cela sortirait certaines liaisons normales, avec un train par jour, comme Marseille → Tourcoing (environ 300 trains par an), alors qu’elle a justement beaucoup de retards. On perdrait des liaisons intéressantes.
Par fiabilité : exclure seulement celles où le taux n’est pas fiable. Avec 100 trains, un retard de plus change le taux de 1 point au maximum, ce qui est acceptable. Ça n’exclut que 3 cas, tous minuscules.
On va prendre la méthode par la fiabilité

In [14]:
volume_min = 100
par_an["volume_min"] = volume_min
par_an.loc[par_an["annee"] == 2018, "volume_min"] = volume_min * 9 / 12

par_an["volume_suffisant"] = par_an["nb_train_prevu"] >= par_an["volume_min"]
print(par_an["volume_suffisant"].value_counts())
print(par_an.loc[~par_an["volume_suffisant"], colonnes])

volume_suffisant
True     1107
False       9
Name: count, dtype: int64
               gare_depart          gare_arrivee  annee  nb_train_prevu  \
319                 MADRID  MARSEILLE ST CHARLES   2020              87   
320                 MADRID  MARSEILLE ST CHARLES   2021               0   
322                 MADRID  MARSEILLE ST CHARLES   2023              67   
362   MARSEILLE ST CHARLES                MADRID   2021               0   
364   MARSEILLE ST CHARLES                MADRID   2023              67   
939        PARIS VAUGIRARD                RENNES   2019              67   
995                 RENNES       PARIS VAUGIRARD   2019              68   
1078             TOURCOING  MARSEILLE ST CHARLES   2019              65   
1079             TOURCOING  MARSEILLE ST CHARLES   2022               2   

      nb_train_retard_arrivee  taux_retard  
319                        20     0.229885  
320                         0          NaN  
322                         9     0.134328 

**Seuils définitifs**
Les seuils de l'étape 2 incluaient les liaisons-années à très faible volume. On les recalcule sur les seules liaisons-années au volume suffisant (au moins 100 trains, 75 pour 2018) : ce sont ces seuils qui servent à la sélection.

In [15]:
# Seuils recalculés sur les liaisons-années au volume suffisant
par_an_ok = par_an[par_an["volume_suffisant"]]

seuils = par_an_ok.groupby("classe_duree", observed=True)["taux_retard"].quantile(0.75)
print(seuils.round(3))

classe_duree
< 1h30     0.181
1h30-3h    0.164
> 3h       0.177
Name: taux_retard, dtype: float64


Seuils inchangés après exclusion des 9 liaisons-années à faible volume : le quartile supérieur est robuste.

### Bilan de l'étape 3 : valeurs extrêmes et volume minimal

**Valeurs extrêmes (classe « > 3h »)**
- Les 2 liaisons-années à 0 % de retard avec au moins 100 trains sont internationales : retards probablement non renseignés.
- Les taux les plus élevés (Paris Est → Stuttgart : jusqu'à 56 %) sont aussi internationaux, avec des volumes importants.
- Les liaisons internationales concentrent les anomalies, en causes (Phase 1) comme en taux de retard : argument pour les exclure de la sélection, ou au moins les signaler.

**Volume minimal : 100 trains par an (75 pour 2018)**
- Critère de stabilité : au-delà de 100 trains, un retard de plus change le taux d'au plus 1 point.
- 9 liaisons-années écartées sur 1 116, toutes marginales : Madrid ↔ Marseille (Covid), Paris Vaugirard ↔ Rennes 2019, Tourcoing → Marseille 2019 et 2022.
- Le seuil de 358 trains de l'exploration venait du découpage des liaisons entre classes. Le 10e percentile corrigé (1 001 trains) exclurait des liaisons régulières à un train par jour, comme Marseille → Tourcoing.

**Seuils définitifs** (après volume minimal) : 18,1 % (< 1h30), 16,4 % (1h30-3h), 17,7 % (> 3h). Inchangés : le quartile supérieur est robuste.

Etape 4 : sélection des liaisons durablement en difficulté.   
La règle du cadrage : une liaison est « durablement en difficulté » si elle est au-dessus du seuil de sa classe la majorité des années de la période de référence. Ici, cela veut dire au moins 2 années sur 3 parmi 2023, 2024 et 2025.

In [16]:
par_an_ok = par_an[par_an["volume_suffisant"]].copy()

par_an_ok["seuil"] = par_an_ok["classe_duree"].map(seuils).astype(float)
par_an_ok["au_dessus"] = par_an_ok["taux_retard"] > par_an_ok["seuil"]

par_an_ok[liaison + ["annee", "classe_duree", "taux_retard", "seuil", "au_dessus"]].head()

,gare_depart,gare_arrivee,annee,classe_duree,taux_retard,seuil,au_dessus
0,AIX EN PROVENCE,PARIS LYON,2018,> 3h,0.120492,0.177326,False
1,AIX EN PROVENCE,PARIS LYON,2019,> 3h,0.106658,0.177326,False
2,AIX EN PROVENCE,PARIS LYON,2020,> 3h,0.087140,0.177326,False
3,AIX EN PROVENCE,PARIS LYON,2021,> 3h,0.078643,0.177326,False
4,AIX EN PROVENCE,PARIS LYON,2022,> 3h,0.125767,0.177326,False


In [17]:
annees_ref = [2023, 2024, 2025]
ref = par_an_ok[par_an_ok["annee"].isin(annees_ref)]

bilan_ref = ref.groupby(liaison).agg(
    nb_annees=("annee", "count"),
    nb_au_dessus=("au_dessus", "sum"),
    classe_duree=("classe_duree", "first"),
)

print(bilan_ref["nb_annees"].value_counts())
print(bilan_ref["nb_au_dessus"].value_counts().sort_index())

nb_annees
3    121
Name: count, dtype: int64
nb_au_dessus
0    82
1    13
2     8
3    18
Name: count, dtype: int64


In [18]:
selection = bilan_ref[bilan_ref["nb_au_dessus"] >= 2].copy()

taux_ref = ref.groupby(liaison).agg(
    nb_train_prevu=("nb_train_prevu", "sum"),
    nb_train_retard_arrivee=("nb_train_retard_arrivee", "sum"),
)
selection["taux_ref"] = taux_ref["nb_train_retard_arrivee"] / taux_ref["nb_train_prevu"]

selection["internationale"] = (
    selection.index.get_level_values("gare_depart").isin(gares_etrangeres)
    | selection.index.get_level_values("gare_arrivee").isin(gares_etrangeres)
)

print(selection["classe_duree"].value_counts())
print(selection["internationale"].value_counts())
selection.sort_values("taux_ref", ascending=False).round(3)

classe_duree
> 3h       12
1h30-3h    11
< 1h30      3
Name: count, dtype: int64
internationale
False    21
True      5
Name: count, dtype: int64


,,nb_annees,nb_au_dessus,classe_duree,taux_ref,internationale
gare_depart,gare_arrivee,,,,,
ITALIE,PARIS LYON,3,3,> 3h,0.272,True
LYON PART DIEU,LILLE,3,3,> 3h,0.270,False
PARIS EST,STUTTGART,3,3,> 3h,0.267,True
LYON PART DIEU,MARNE LA VALLEE,3,3,1h30-3h,0.253,False
MACON LOCHE,PARIS LYON,3,3,1h30-3h,0.251,False
CHAMBERY CHALLES LES EAUX,PARIS LYON,3,3,1h30-3h,0.238,False
LYON PART DIEU,MARSEILLE ST CHARLES,3,3,1h30-3h,0.220,False
MARSEILLE ST CHARLES,LILLE,3,3,> 3h,0.217,False
ARRAS,PARIS NORD,3,3,< 1h30,0.214,False


### Bilan de l'étape 4 : liaisons durablement en difficulté

**Règle** : au-dessus du seuil de sa classe au moins 2 années sur 3 de la période de référence (2023, 2024, 2025 ; années d'octobre à septembre).

**Résultat** : 26 liaisons sur 121 présentes sur la période (21 %). 18 sont au-dessus du seuil les 3 années. 9 liaisons n'apparaissent pas sur la période de référence.
- Par classe : 12 de plus de 3 h, 11 de 1 h 30 à 3 h, 3 de moins de 1 h 30.
- 5 sont internationales, dont la première (Italie → Paris Lyon, 27 %) : données suspectes (voir étape 3), à signaler. Sans elles : 21 liaisons.

**Observations**
- Lyon Part-Dieu figure dans 7 liaisons, surtout province-province (Lille, Marne-la-Vallée, Montpellier, Marseille).
- L'axe Paris-Sud-Est est très présent (Mâcon, Chambéry, Valence, Nîmes, Annecy).
- Plusieurs liaisons sont sélectionnées dans les deux sens : la difficulté touche des axes plutôt qu'un sens.
- Cas limite : Bordeaux → Tourcoing, au-dessus du seuil 2 années sur 3, mais avec un taux sur la période (15,7 %) sous le seuil.

Étape 5 : l’évolution sur la période de suivi

In [19]:
suivi = par_an_ok[par_an_ok["annee"] == 2026].set_index(liaison)

selection["taux_suivi"] = suivi["taux_retard"]
selection["au_dessus_2026"] = suivi["au_dessus"]
selection["evolution"] = selection["taux_suivi"] - selection["taux_ref"]

print(selection["taux_suivi"].isna().sum(), "liaison(s) sans donnée en 2026")

0 liaison(s) sans donnée en 2026


In [20]:
print((selection["evolution"] > 0.01).sum(), "se dégradent (plus de 1 point)")
print((selection["evolution"].abs() <= 0.01).sum(), "stables (à 1 point près)")
print((selection["evolution"] < -0.01).sum(), "s'améliorent (plus de 1 point)")
print(selection["au_dessus_2026"].sum(), "toujours au-dessus du seuil en 2026")

colonnes_suivi = ["classe_duree", "internationale", "taux_ref", "taux_suivi", "evolution", "au_dessus_2026"]
selection[colonnes_suivi].sort_values("evolution", ascending=False).round(3)

23 se dégradent (plus de 1 point)
3 stables (à 1 point près)
0 s'améliorent (plus de 1 point)
25 toujours au-dessus du seuil en 2026


classe_duree  \
gare_depart               gare_arrivee                             
PARIS EST                 STUTTGART                         > 3h   
                          FRANCFORT                         > 3h   
ITALIE                    PARIS LYON                        > 3h   
CHAMBERY CHALLES LES EAUX PARIS LYON                     1h30-3h   
MACON LOCHE               PARIS LYON                     1h30-3h   
MARNE LA VALLEE           MARSEILLE ST CHARLES              > 3h   
LYON PART DIEU            MARNE LA VALLEE                1h30-3h   
DOUAI                     PARIS NORD                      < 1h30   
LILLE                     MARSEILLE ST CHARLES              > 3h   
PARIS LYON                CHAMBERY CHALLES LES EAUX      1h30-3h   
LYON PART DIEU            MARSEILLE ST CHARLES           1h30-3h   
ARRAS                     PARIS NORD                      < 1h30   
ANNECY                    PARIS LYON                        > 3h   
LYON PART DIEU            MONTPELLIER                    1h30-3h   
BORDEAUX ST JEAN          TOURCOING                         > 3h   
LE MANS                   PARIS MONTPARNASSE              < 1h30   
STRASBOURG                PARIS EST                      1h30-3h   
NIMES                     PARIS LYON                     1h30-3h   
VALENCE ALIXAN            PARIS LYON                     1h30-3h   
STUTTGART                 PARIS EST                         > 3h   
MARNE LA VALLEE           LYON PART DIEU                 1h30-3h   
BARCELONA                 PARIS LYON                        > 3h   
MARSEILLE ST CHARLES      LILLE                             > 3h   
MONTPELLIER               LYON PART DIEU                 1h30-3h   
LYON PART DIEU            LILLE                             > 3h   
LILLE                     LYON PART DIEU                    > 3h   

                                                     internationale  taux_ref  \
gare_depart               gare_arrivee                                          
PARIS EST                 STUTTGART                            True     0.267   
                          FRANCFORT                            True     0.213   
ITALIE                    PARIS LYON                           True     0.272   
CHAMBERY CHALLES LES EAUX PARIS LYON                          False     0.238   
MACON LOCHE               PARIS LYON                          False     0.251   
MARNE LA VALLEE           MARSEILLE ST CHARLES                False     0.185   
LYON PART DIEU            MARNE LA VALLEE                     False     0.253   
DOUAI                     PARIS NORD                          False     0.211   
LILLE                     MARSEILLE ST CHARLES                False     0.190   
PARIS LYON                CHAMBERY CHALLES LES EAUX           False     0.189   
LYON PART DIEU            MARSEILLE ST CHARLES                False     0.220   
ARRAS                     PARIS NORD                          False     0.214   
ANNECY                    PARIS LYON                          False     0.176   
LYON PART DIEU            MONTPELLIER                         False     0.207   
BORDEAUX ST JEAN          TOURCOING                           False     0.157   
LE MANS                   PARIS MONTPARNASSE                  False     0.182   
STRASBOURG                PARIS EST                           False     0.185   
NIMES                     PARIS LYON                          False     0.182   
VALENCE ALIXAN            PARIS LYON                          False     0.185   
STUTTGART                 PARIS EST                            True     0.198   
MARNE LA VALLEE           LYON PART DIEU                      False     0.196   
BARCELONA                 PARIS LYON                           True     0.181   
MARSEILLE ST CHARLES      LILLE                               False     0.217   
MONTPELLIER               LYON PART DIEU                      False     0.181   
LYON PART DIEU            LILLE          

In [21]:
taux_ref_tous = ref.groupby(liaison)["nb_train_retard_arrivee"].sum() / ref.groupby(liaison)["nb_train_prevu"].sum()
evolution_tous = suivi["taux_retard"] - taux_ref_tous

est_selectionnee = evolution_tous.index.isin(selection.index)

print("Liaisons sélectionnées :", evolution_tous[est_selectionnee].median().round(3))
print("Autres liaisons        :", evolution_tous[~est_selectionnee].median().round(3))
print((evolution_tous[~est_selectionnee] > 0.01).mean().round(2), "des autres liaisons se dégradent aussi")

Liaisons sélectionnées : 0.047
Autres liaisons        : 0.04
0.97 des autres liaisons se dégradent aussi


In [22]:
par_mois = df[df["date"] >= "2023-01"].copy()
par_mois["taux"] = par_mois["nb_train_retard_arrivee"] / par_mois["nb_train_prevu"]

mediane_mois = par_mois.groupby("date")["taux"].median()
print(mediane_mois.round(3).to_string())

date
2023-01    0.115
2023-02    0.092
2023-03    0.105
2023-04    0.133
2023-05    0.121
2023-06    0.168
2023-07    0.156
2023-08    0.146
2023-09    0.141
2023-10    0.145
2023-11    0.155
2023-12    0.133
2024-01    0.144
2024-02    0.107
2024-03    0.130
2024-04    0.127
2024-05    0.103
2024-06    0.128
2024-07    0.149
2024-08    0.097
2024-09    0.133
2024-10    0.114
2024-11    0.157
2024-12    0.127
2025-01    0.139
2025-02    0.117
2025-03    0.110
2025-04    0.106
2025-05    0.101
2025-06    0.156
2025-07    0.167
2025-08    0.131
2025-09    0.147
2025-10    0.150
2025-11    0.174
2025-12    0.163
2026-01    0.215
2026-02    0.146
2026-03    0.100
2026-04    0.142
2026-05    0.183
2026-06    0.230
2026-07    0.232
2026-08    0.186
2026-09    0.149


**2026 est une année dégradée pour tout le réseau** (+4 points de taux de retard pour la liaison médiane non sélectionnée ; 97 % des liaisons se dégradent). La hausse est concentrée sur janvier et mai-août 2026 ; les autres mois restent dans la norme, ce qui écarte un changement de méthode de comptage. On mesure donc l'évolution des liaisons sélectionnées par rapport à cette tendance générale.

In [23]:
tendance = evolution_tous[~est_selectionnee].median()

selection["evolution_relative"] = selection["evolution"] - tendance

print("Tendance du réseau :", round(tendance, 3))
print((selection["evolution_relative"] > 0.01).sum(), "se dégradent plus que le réseau")
print((selection["evolution_relative"].abs() <= 0.01).sum(), "suivent le réseau (à 1 point près)")
print((selection["evolution_relative"] < -0.01).sum(), "se dégradent moins que le réseau")

selection[["taux_ref", "taux_suivi", "evolution", "evolution_relative", "internationale"]].sort_values("evolution_relative", ascending=False).round(3)

Tendance du réseau : 0.04
12 se dégradent plus que le réseau
7 suivent le réseau (à 1 point près)
7 se dégradent moins que le réseau


taux_ref  taux_suivi  \
gare_depart               gare_arrivee                                      
PARIS EST                 STUTTGART                     0.267       0.565   
                          FRANCFORT                     0.213       0.324   
ITALIE                    PARIS LYON                    0.272       0.365   
CHAMBERY CHALLES LES EAUX PARIS LYON                    0.238       0.325   
MACON LOCHE               PARIS LYON                    0.251       0.337   
MARNE LA VALLEE           MARSEILLE ST CHARLES          0.185       0.254   
LYON PART DIEU            MARNE LA VALLEE               0.253       0.315   
DOUAI                     PARIS NORD                    0.211       0.270   
LILLE                     MARSEILLE ST CHARLES          0.190       0.249   
PARIS LYON                CHAMBERY CHALLES LES EAUX     0.189       0.247   
LYON PART DIEU            MARSEILLE ST CHARLES          0.220       0.276   
ARRAS                     PARIS NORD                    0.214       0.268   
ANNECY                    PARIS LYON                    0.176       0.223   
LYON PART DIEU            MONTPELLIER                   0.207       0.253   
BORDEAUX ST JEAN          TOURCOING                     0.157       0.202   
LE MANS                   PARIS MONTPARNASSE            0.182       0.225   
STRASBOURG                PARIS EST                     0.185       0.223   
NIMES                     PARIS LYON                    0.182       0.218   
VALENCE ALIXAN            PARIS LYON                    0.185       0.219   
STUTTGART                 PARIS EST                     0.198       0.222   
MARNE LA VALLEE           LYON PART DIEU                0.196       0.218   
BARCELONA                 PARIS LYON                    0.181       0.199   
MARSEILLE ST CHARLES      LILLE                         0.217       0.234   
MONTPELLIER               LYON PART DIEU                0.181       0.190   
LYON PART DIEU            LILLE                         0.270       0.271   
LILLE                     LYON PART DIEU                0.180       0.176   

                                                     evolution  \
gare_depart               gare_arrivee                           
PARIS EST                 STUTTGART                      0.298   
                          FRANCFORT                      0.110   
ITALIE                    PARIS LYON                     0.093   
CHAMBERY CHALLES LES EAUX PARIS LYON                     0.086   
MACON LOCHE               PARIS LYON                     0.085   
MARNE LA VALLEE           MARSEILLE ST CHARLES           0.069   
LYON PART DIEU            MARNE LA VALLEE                0.062   
DOUAI                     PARIS NORD                     0.060   
LILLE                     MARSEILLE ST CHARLES           0.059   
PARIS LYON                CHAMBERY CHALLES LES EAUX      0.058   
LYON PART DIEU            MARSEILLE ST CHARLES           0.057   
ARRAS                     PARIS NORD                     0.054   
ANNECY                    PARIS LYON                     0.047   
LYON PART DIEU            MONTPELLIER                    0.046   
BORDEAUX ST JEAN          TOURCOING                      0.046   
LE MANS                   PARIS MONTPARNASSE             0.043   
STRASBOURG                PARIS EST                      0.037   
NIMES                     PARIS LYON                     0.036   
VALENCE ALIXAN            PARIS LYON                     0.034   
STUTTGART                 PARIS EST                      0.024   
MARNE LA VALLEE           LYON PART DIEU                 0.022   
BARCELONA                 PARIS LYON                     0.018   
MARSEILLE ST CHARLES      LILLE                          0.016   
MONTPELLIER               LYON PART DIEU                 0.010   
LYON PART DIEU            LILLE                          0.001   
LILLE                     LYON PART DIEU                -0.003   

                                               

### Bilan de l'étape 5A : évolution sur la période de suivi (oct. 2025 → sept. 2026)

- **Année dégradée pour tout le réseau** : +4 points de taux de retard pour la liaison médiane non sélectionnée, 97 % des liaisons se dégradent. La hausse est concentrée sur janvier et mai-août 2026 ; les autres mois restent dans la norme (pas de changement de méthode de comptage apparent).
- Les 26 liaisons sélectionnées se dégradent en médiane de +4,7 points, soit à peine plus que le réseau. On mesure donc leur évolution **relative** (écart à la tendance du réseau).
- **Évolution relative** : 12 se dégradent plus que le réseau (9 françaises), 7 suivent le réseau, 7 se dégradent moins (5 françaises).
- Les 3 plus fortes dégradations sont internationales (Paris Est → Stuttgart : 56,5 % en 2026), avec des données suspectes (étape 3).
- Liaisons françaises qui décrochent : surtout vers Paris Lyon (Chambéry, Mâcon : plus de 32 % en 2026), plus nettement que dans le sens inverse ; aussi Marne-la-Vallée → Marseille, Lyon → Marne-la-Vallée, et les liaisons courtes du Nord (Douai, Arras → Paris Nord).
- Liaisons qui résistent : Lyon ↔ Lille (Lyon → Lille stable à 27 %), Montpellier → Lyon, Marseille → Lille. Une partie de cette résistance relève de la régression vers la moyenne.

Étape 5B : quelle cause porte l’évolution ?

In [24]:
familles = {
    "reseau": ["prct_cause_infra", "prct_cause_gestion_trafic"],
    "voyageurs": ["prct_cause_materiel_roulant", "prct_cause_gestion_gare", "prct_cause_prise_en_charge_voyageurs"],
    "externe": ["prct_cause_externe"],
}

# Même règle qu'en Phase 1 : des trains en retard, mais une somme des causes à plus de 5 points de 100
toutes_causes = familles["reseau"] + familles["voyageurs"] + familles["externe"]
somme_causes = df[toutes_causes].sum(axis=1)
a_des_retards = df["nb_train_retard_arrivee"] > 0
df["cause_fiable"] = ~(a_des_retards & ((somme_causes - 100).abs() > 5))
print(df["cause_fiable"].value_counts())

for famille, colonnes in familles.items():
    df[f"nb_retard_{famille}"] = df["nb_train_retard_arrivee"] * df[colonnes].sum(axis=1) / 100

cause_fiable
True     12824
False       83
Name: count, dtype: int64


In [25]:
df["periode"] = None
df.loc[df["annee"].isin(annees_ref), "periode"] = "reference"
df.loc[df["annee"] == 2026, "periode"] = "suivi"

df_causes = df[df["cause_fiable"] & df["periode"].notna()]

contrib = df_causes.groupby(liaison + ["periode"]).agg(
    nb_train_prevu=("nb_train_prevu", "sum"),
    nb_retard_reseau=("nb_retard_reseau", "sum"),
    nb_retard_voyageurs=("nb_retard_voyageurs", "sum"),
    nb_retard_externe=("nb_retard_externe", "sum"),
)

for famille in familles:
    contrib[f"contrib_{famille}"] = contrib[f"nb_retard_{famille}"] / contrib["nb_train_prevu"]

print(contrib.shape)
contrib.head(4).round(3)

(244, 7)


nb_train_prevu  \
gare_depart     gare_arrivee       periode                     
AIX EN PROVENCE PARIS LYON         reference           16668   
                                   suivi                5200   
ANGERS ST LAUD  PARIS MONTPARNASSE reference           19465   
                                   suivi                6569   

                                              nb_retard_reseau  \
gare_depart     gare_arrivee       periode                       
AIX EN PROVENCE PARIS LYON         reference           941.018   
                                   suivi               202.343   
ANGERS ST LAUD  PARIS MONTPARNASSE reference          1235.994   
                                   suivi               309.689   

                                              nb_retard_voyageurs  \
gare_depart     gare_arrivee       periode                          
AIX EN PROVENCE PARIS LYON         reference              675.311   
                                   suivi                  413.800   
ANGERS ST LAUD  PARIS MONTPARNASSE reference              755.354   
                                   suivi                  389.164   

                                              nb_retard_externe  \
gare_depart     gare_arrivee       periode                        
AIX EN PROVENCE PARIS LYON         reference            372.671   
                                   suivi                278.857   
ANGERS ST LAUD  PARIS MONTPARNASSE reference            620.651   
                                   suivi                467.147   

                                              contrib_reseau  \
gare_depart     gare_arrivee       periode                     
AIX EN PROVENCE PARIS LYON         reference           0.056   
                                   suivi               0.039   
ANGERS ST LAUD  PARIS MONTPARNASSE reference           0.063   
                                   suivi               0.047   

                                              contrib_voyageurs  \
gare_depart     gare_arrivee       periode                        
AIX EN PROVENCE PARIS LYON         reference              0.041   
                                   suivi                  0.080   
ANGERS ST LAUD  PARIS MONTPARNASSE reference              0.039   
                                   suivi                  0.059   

                                              contrib_externe  
gare_depart     gare_arrivee       periode                     
AIX EN PROVENCE PARIS LYON         reference            0.022  
                                   suivi                0.054  
ANGERS ST LAUD  PARIS MONTPARNASSE reference            0.032  
                                   suivi                0.071

In [26]:
colonnes_contrib = ["contrib_reseau", "contrib_voyageurs", "contrib_externe"]

c_ref = contrib.xs("reference", level="periode")[colonnes_contrib]
c_suivi = contrib.xs("suivi", level="periode")[colonnes_contrib]

delta = (c_suivi - c_ref).dropna()
delta["est_selectionnee"] = delta.index.isin(selection.index)

print(delta.groupby("est_selectionnee").median().round(3))

                  contrib_reseau  contrib_voyageurs  contrib_externe
est_selectionnee                                                    
False                     -0.011              0.028            0.022
True                      -0.026              0.030            0.027


### Bilan de l'étape 5B : causes de l'évolution

Évolution médiane des contributions entre la référence (2023-2025) et le suivi (2026), en points de taux de retard :

| | Réseau | Voyageurs | Externe |
|---|---|---|---|
| Autres liaisons | −1,1 | +2,8 | +2,2 |
| Liaisons sélectionnées | −2,6 | +3,0 | +2,7 |

- La dégradation de 2026 est portée par les Voyageurs et l'Externe, à parts comparables. La contribution du Réseau baisse.
- Les liaisons sélectionnées suivent le même schéma, avec une baisse du Réseau plus marquée.
- Hypothèse à vérifier : les mois les plus touchés (janvier, mai-août) évoquent la météo, qui pèse sur l'Externe et le matériel roulant.

**Limites**
- Les médianes de chaque famille viennent de liaisons différentes : elles donnent des tendances, pas une décomposition exacte.
- Une seule cause est retenue par train : une hausse des incidents externes peut faire baisser mécaniquement la part attribuée au Réseau.

In [27]:
mediane_autres = delta.loc[~delta["est_selectionnee"], colonnes_contrib].median()

delta_sel = delta.loc[delta["est_selectionnee"], colonnes_contrib]
ecart = delta_sel - mediane_autres

resultat = pd.DataFrame({
    "cause_hausse": delta_sel.idxmax(axis=1),
    "cause_propre": ecart.idxmax(axis=1),
})
resultat = resultat.join(selection[["evolution_relative", "internationale"]])

print(resultat["cause_hausse"].value_counts())
print(resultat["cause_propre"].value_counts())
resultat.sort_values("evolution_relative", ascending=False).round(3)

cause_hausse
contrib_voyageurs    14
contrib_externe      11
contrib_reseau        1
Name: count, dtype: int64
cause_propre
contrib_externe      14
contrib_voyageurs     6
contrib_reseau        6
Name: count, dtype: int64


cause_hausse  \
gare_depart               gare_arrivee                                   
PARIS EST                 STUTTGART                    contrib_externe   
                          FRANCFORT                    contrib_externe   
ITALIE                    PARIS LYON                 contrib_voyageurs   
CHAMBERY CHALLES LES EAUX PARIS LYON                 contrib_voyageurs   
MACON LOCHE               PARIS LYON                 contrib_voyageurs   
MARNE LA VALLEE           MARSEILLE ST CHARLES       contrib_voyageurs   
LYON PART DIEU            MARNE LA VALLEE            contrib_voyageurs   
DOUAI                     PARIS NORD                 contrib_voyageurs   
LILLE                     MARSEILLE ST CHARLES       contrib_voyageurs   
PARIS LYON                CHAMBERY CHALLES LES EAUX  contrib_voyageurs   
LYON PART DIEU            MARSEILLE ST CHARLES         contrib_externe   
ARRAS                     PARIS NORD                 contrib_voyageurs   
ANNECY                    PARIS LYON                   contrib_externe   
LYON PART DIEU            MONTPELLIER                contrib_voyageurs   
BORDEAUX ST JEAN          TOURCOING                  contrib_voyageurs   
LE MANS                   PARIS MONTPARNASSE           contrib_externe   
STRASBOURG                PARIS EST                    contrib_externe   
NIMES                     PARIS LYON                   contrib_externe   
VALENCE ALIXAN            PARIS LYON                   contrib_externe   
STUTTGART                 PARIS EST                     contrib_reseau   
MARNE LA VALLEE           LYON PART DIEU             contrib_voyageurs   
BARCELONA                 PARIS LYON                   contrib_externe   
MARSEILLE ST CHARLES      LILLE                      contrib_voyageurs   
MONTPELLIER               LYON PART DIEU             contrib_voyageurs   
LYON PART DIEU            LILLE                        contrib_externe   
LILLE                     LYON PART DIEU               contrib_externe   

                                                          cause_propre  \
gare_depart               gare_arrivee                                   
PARIS EST                 STUTTGART                    contrib_externe   
                          FRANCFORT                    contrib_externe   
ITALIE                    PARIS LYON                    contrib_reseau   
CHAMBERY CHALLES LES EAUX PARIS LYON                    contrib_reseau   
MACON LOCHE               PARIS LYON                    contrib_reseau   
MARNE LA VALLEE           MARSEILLE ST CHARLES       contrib_voyageurs   
LYON PART DIEU            MARNE LA VALLEE            contrib_voyageurs   
DOUAI                     PARIS NORD                 contrib_voyageurs   
LILLE                     MARSEILLE ST CHARLES         contrib_externe   
PARIS LYON                CHAMBERY CHALLES LES EAUX     contrib_reseau   
LYON PART DIEU            MARSEILLE ST CHARLES         contrib_externe   
ARRAS                     PARIS NORD                 contrib_voyageurs   
ANNECY                    PARIS LYON                   contrib_externe   
LYON PART DIEU            MONTPELLIER                  contrib_externe   
BORDEAUX ST JEAN          TOURCOING                  contrib_voyageurs   
LE MANS                   PARIS MONTPARNASSE           contrib_externe   
STRASBOURG                PARIS EST                     contrib_reseau   
NIMES                     PARIS LYON                   contrib_externe   
VALENCE ALIXAN            PARIS LYON                   contrib_externe   
STUTTGART                 PARIS EST                     contrib_reseau   
MARNE LA VALLEE           LYON PART DIEU               contrib_externe   
BARCELONA                 PARIS LYON                   contrib_externe   
MARSEILLE ST CHARLES      LILLE                      contrib_voyageurs   
MONTPELLIER               LYON PART DIEU               contrib_externe   
LYON PART DIEU            LILLE                        contri

### Bilan de l'étape 5B (suite) : la cause propre à chaque liaison

On compare, pour chaque liaison sélectionnée, l'évolution de chaque famille à la tendance du réseau (médiane des liaisons non sélectionnées).

- Ce qui augmente le plus : les Voyageurs (14 liaisons) et l'Externe (11), comme pour tout le réseau. Le Réseau seulement pour 1 liaison.
- Parmi les 9 liaisons françaises qui se dégradent plus que le réseau :
  - **Réseau** : Chambéry → Paris Lyon, Mâcon → Paris Lyon, Paris Lyon → Chambéry. Ce sont les trois plus forts décrochages français, et Italie → Paris Lyon (même couloir) présente la même cause propre. Problème localisé sur l'axe Paris – Mâcon – Chambéry, alors que le Réseau recule ailleurs. Origine à vérifier (travaux, incidents, saturation).
  - **Voyageurs** : Marne-la-Vallée → Marseille, Lyon → Marne-la-Vallée, Douai → Paris Nord, Arras → Paris Nord.
  - **Externe** : Lille → Marseille, Lyon → Marseille.
- Pour les liaisons qui se dégradent moins que le réseau, la « cause propre » n'est pas interprétée.